In [3]:
# --- 0. INSTALL PIPER ---
!pip install -q piper-tts

import os
import json
import subprocess
import shutil
from huggingface_hub import hf_hub_download

# --- 1. DOWNLOAD FILES ---
print("Downloading Hindi Model & Config...")
repo = "rhasspy/piper-voices"

# Using 'priyamvada' (Official Hindi Female Voice)
onnx_file = "hi/hi_IN/priyamvada/medium/hi_IN-priyamvada-medium.onnx"
json_file = "hi/hi_IN/priyamvada/medium/hi_IN-priyamvada-medium.onnx.json"

onnx_path = hf_hub_download(repo_id=repo, filename=onnx_file)
json_path = hf_hub_download(repo_id=repo, filename=json_file)

# Ensure they are in the same directory and named correctly for Piper
model_dir = "/content/piper_model"
os.makedirs(model_dir, exist_ok=True)

final_onnx = os.path.join(model_dir, "model.onnx")
final_json = os.path.join(model_dir, "model.onnx.json")

shutil.copy(onnx_path, final_onnx)
shutil.copy(json_path, final_json)

# --- 2. EXECUTION ---
# Ensure this matches the name of your uploaded JSON file
JSON_PATH = "/content/hindi_evaluation_set.json"
OUTPUT_DIR = "/content/output_audio/hindi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print("\n🚀 Starting Piper TTS Generation for HINDI...")
print("-" * 50)

for key, item in dataset.items():
    item_id = item["id"]
    text = item["text"]
    filename = f"{OUTPUT_DIR}/{item_id}.wav"

    print(f"Processing : {item_id}")

    # FIXED: Using 'python -m piper' ensures it runs regardless of PATH issues
    process = subprocess.Popen(
        ["python", "-m", "piper", "--model", final_onnx, "--output_file", filename],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    stdout, stderr = process.communicate(input=text.encode('utf-8'))

    if process.returncode == 0:
        print(f"✅ Success: {filename}")
    else:
        print(f"❌ FAILED: {stderr.decode('utf-8')}")

print("-" * 50)
print("Hindi Piper generation complete! You can find the files in the 'output_audio/hindi' folder.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 87.5 MB/s eta 0:00:00

🚀 Starting Piper TTS Generation for HINDI...
--------------------------------------------------
Processing : HIN_01
✅ Success: /content/output_audio/hindi/HIN_01.wav
Processing : HIN_02
✅ Success: /content/output_audio/hindi/HIN_02.wav
Processing : HIN_03
✅ Success: /content/output_audio/hindi/HIN_03.wav
Processing : HIN_04
✅ Success: /content/output_audio/hindi/HIN_04.wav
Processing : HIN_05
✅ Success: /content/output_audio/hindi/HIN_05.wav
Processing : HIN_06
✅ Success: /content/output_audio/hindi/HIN_06.wav
Processing : HIN_07
✅ Success: /content/output_audio/hindi/HIN_07.wav
Processing : HIN_08
✅ Success: /content/output_audio/hindi/HIN_08.wav
Processing : HIN_09
✅ Success: /content/output_audio/hindi/HIN_09.wav
Processing : HIN_10
✅ Success: /content/output_audio/hindi/HIN_10.wav
Processing : HIN_11
✅ Success: /content/outp

In [5]:
!pip install -q openai-whisper jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 42.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.6 MB/s eta 0:00:00


In [6]:
import os
import json
import torch
import librosa
import shutil
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Change these to match the language you are currently evaluating!
TARGET_LANGUAGE = "hindi" # Options: "hindi", "assamese", "bengali", "nepali"
AUDIO_DIR = f"/content/output_audio/{TARGET_LANGUAGE}"
JSON_PATH = f"/content/{TARGET_LANGUAGE}_evaluation_set.json"

# Set the text key based on the JSON structure of that language
if TARGET_LANGUAGE == "hindi":
    text_key = "text"
elif TARGET_LANGUAGE == "assamese":
    text_key = "assamese_sentence"
elif TARGET_LANGUAGE == "bengali":
    text_key = "bengali_sentence"
elif TARGET_LANGUAGE == "nepali":
    text_key = "TEXT" # Based on the earlier Nepali JSON we made

# ==========================================
# 2. LOAD WHISPER MEDIUM
# ==========================================
print(f"Loading Whisper Medium ASR model for {TARGET_LANGUAGE.upper()}...")
device = "cuda" if torch.cuda.is_available() else "cpu"

asr_model_id = "openai/whisper-medium"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id).to(device)

# Force Whisper to transcribe in the correct language
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language=TARGET_LANGUAGE, task="transcribe")

# ==========================================
# 3. LOAD GROUND TRUTH DATA
# ==========================================
try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)

    # Handle the two different JSON structures (List vs Dictionary)
    if isinstance(dataset, dict):
        ground_truth = {str(item.get("id", k)): item.get(text_key, "") for k, item in dataset.items()}
    else:
        ground_truth = {str(item.get("id", item.get("ID"))): item.get(text_key, "") for item in dataset}

    print(f"Loaded {len(ground_truth)} reference sentences.")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{JSON_PATH}'.")
    ground_truth = {}

# ==========================================
# 4. EVALUATION LOOP (WER & CER)
# ==========================================
print(f"\n🚀 Starting Whisper Evaluation for {TARGET_LANGUAGE.upper()}...")
print("-" * 60)

results = []

# Iterate through the generated audio files
for filename in os.listdir(AUDIO_DIR):
    if not filename.endswith(".wav"):
        continue

    file_path = os.path.join(AUDIO_DIR, filename)

    # Extract ID from filename (Assuming format like "HIN_01.wav" or "hindi_HIN_01.wav")
    # This logic safely grabs the ID regardless of prefix
    base_name = filename.replace(".wav", "").replace(f"{TARGET_LANGUAGE}_", "")
    ref_text = ground_truth.get(base_name, "")

    if not ref_text:
        continue

    # A. Transcribe Audio
    speech_array, sampling_rate = librosa.load(file_path, sr=16000)
    input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # B. Normalize Text (Remove punctuation that ruins WER calculation)
    norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
    norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

    # C. Calculate Metrics
    try:
        item_wer = wer(norm_ref, norm_hyp)
        item_cer = cer(norm_ref, norm_hyp)
    except ValueError:
        item_wer, item_cer = 1.0, 1.0 # Failsafe for empty strings

    print(f"File: {filename}")
    print(f"  Ref: {norm_ref}")
    print(f"  Hyp: {norm_hyp}")
    print(f"  --> WER: {item_wer:.3f} | CER: {item_cer:.3f}\n")

    results.append({"file": filename, "wer": item_wer, "cer": item_cer})

# Calculate Averages
if results:
    avg_wer = sum(r["wer"] for r in results) / len(results)
    avg_cer = sum(r["cer"] for r in results) / len(results)
    print("-" * 60)
    print(f"🎯 AVERAGE METRICS FOR {TARGET_LANGUAGE.upper()}: WER = {avg_wer:.3f} | CER = {avg_cer:.3f}")
    print("-" * 60)

# ==========================================
# 5. ZIP AND DOWNLOAD
# ==========================================
if os.path.exists(AUDIO_DIR) and len(os.listdir(AUDIO_DIR)) > 0:
    zip_filename = f"{TARGET_LANGUAGE}_evaluation_audio"
    print(f"\n📦 Zipping the '{AUDIO_DIR}' directory...")

    # Create the zip file
    shutil.make_archive(zip_filename, 'zip', AUDIO_DIR)

    print("⬇️ Triggering download to your local machine...")
    # Trigger browser download
    files.download(f"{zip_filename}.zip")
else:
    print(f"⚠️ No audio files found in {AUDIO_DIR} to zip.")

Loading Whisper Medium ASR model for HINDI...


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

Loaded 20 reference sentences.

🚀 Starting Whisper Evaluation for HINDI...
------------------------------------------------------------


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressToken

File: HIN_16.wav
  Ref: वाह तुमने तो कमाल कर दिया; लेकिन क्या तुम्हें सच में लगता है कि यह मुमकिन है
  Hyp: वार् तुमने तो कमाल कर दिया लेकिन क्या तुमहे सचमी लगता है कि यह है मुंकिन है
  --> WER: 0.412 | CER: 0.158

File: HIN_06.wav
  Ref: ज़रा फ़िक्र मत करो क़लम से ख़त लिखकर ग़ज़ल का मज़ा लो
  Hyp: जरा फेकर मत करो कलम से खत लिककर गद का मजा लो
  --> WER: 0.583 | CER: 0.226

File: HIN_02.wav
  Ref: कमल और काव्या ने खेत से कद्दू उखाड़ा फिर गरम घी का घड़ा उठाकर घर की ओर भागे
  Hyp: कमल और काव्याने खेच से कदू खाडा फिर गरम घी का घड़ा उठाकर घर की और भागे
  --> WER: 0.333 | CER: 0.093

File: HIN_05.wav
  Ref: भालू ने भारी पेड़ पर बैठकर मीठा फल खाया और पानी पी लिया
  Hyp: भालू ने भारी पेड पर बैटकर मीथा फल खाया और पानी पी लिया
  --> WER: 0.231 | CER: 0.055

File: HIN_01.wav
  Ref: एक आम आदमी और औरत इमली के पेड़ के नीचे बैठकर ऊन बुन रहे हैं
  Hyp: एक आम आदमी और औरत इम्ली के पेड के निचे बैट कर उन बुन रहे हैं
  --> WER: 0.400 | CER: 0.102

File: HIN_03.wav
  Ref: षट्कोण के अंदर रखे ढक्कन और डमरू को

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>